In [2]:
!pip install langchain_huggingface


[notice] A new release of pip is available: 25.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
!pip install tf-keras


[notice] A new release of pip is available: 25.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
# from langchain.document_loaders import TextLoader
# from langchain.text_splitter import CharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
from langchain_core.runnables import RunnableParallel
import os

c:\Users\HP\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"

In [7]:
llm = ChatGroq(
    api_key=groq_api_key,  
    model="llama-3.3-70b-versatile",
    temperature=.7
)

In [10]:
loader = TextLoader("sample_faq.txt", encoding="utf-8")
documents = loader.load()

text_splitter = CharacterTextSplitter(chunk_size=300, chunk_overlap=50)
docs = text_splitter.split_documents(documents)

In [12]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(docs, embeddings)
# retriever = vectorstore.as_retriever()

In [15]:
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 4, "fetch_k": 20, "lambda_mult": 0.5}
)

In [16]:
prompt = ChatPromptTemplate.from_template(
    """Answer the question based only on the following context.
If you don't know the answer, say you don't know.

Context:
{context}

Question: {question}"""
)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)


In [17]:
# qa_chain = RetrievalQA.from_chain_type(
#     llm=llm,
#     retriever=vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 2}),
#     return_source_documents=True
# )

qa_chain = RunnableParallel(
    {"context": retriever, "question": RunnablePassthrough()}
) | RunnablePassthrough.assign(
    answer=(
        {"context": lambda x: format_docs(x["context"]), "question": lambda x: x["question"]}
        | prompt | llm | StrOutputParser()
    )
)

In [ ]:
response = qa_chain.invoke("What is LangChain used for?")
print("Answer:", response["answer"])
print("\nSources:")
for doc in response["context"]:
    print("-", doc.page_content)

Answer: I don't know.

Sources:
- Premium Plan Benefits:
The premium plan includes priority customer support, access to exclusive webinars, and a dedicated success manager for enterprise users.
- Refund Policy:
We offer a 30-day money-back guarantee on all our plans. If you're not satisfied, contact support within 30 days for a full refund.

Shipping Duration:
Shipping typically takes between 5-7 business days, depending on your location. International orders may take longer.
- Subscription Cancellation:
Customers can cancel their subscription anytime through the account settings page. No additional charges will apply after cancellation.


In [31]:
response = qa_chain.invoke("What is LangChain used for?")
print("Answer:", response["answer"])
print("\nSources:")
for doc in response["context"]:
    print("-", doc.page_content)

Answer: I don't know.

Sources:
- Premium Plan Benefits:
The premium plan includes priority customer support, access to exclusive webinars, and a dedicated success manager for enterprise users.
- Refund Policy:
We offer a 30-day money-back guarantee on all our plans. If you're not satisfied, contact support within 30 days for a full refund.

Shipping Duration:
Shipping typically takes between 5-7 business days, depending on your location. International orders may take longer.
- Subscription Cancellation:
Customers can cancel their subscription anytime through the account settings page. No additional charges will apply after cancellation.


In [32]:
questions = [
    "What is the refund policy?",
    "How long does shipping usually take?",
    "Can customers cancel their subscription anytime?",
    "What are the benefits of the premium plan?"
]
for q in questions:
    ques="Query: "+q
    response = qa_chain.invoke(ques)
    print("Answer:", response["answer"])
    print("\nSources:")
    for doc in response["context"]:
        print("-", doc.page_content)

Answer: The refund policy is a 30-day money-back guarantee. If you're not satisfied, you can contact support within 30 days for a full refund.

Sources:
- Refund Policy:
We offer a 30-day money-back guarantee on all our plans. If you're not satisfied, contact support within 30 days for a full refund.

Shipping Duration:
Shipping typically takes between 5-7 business days, depending on your location. International orders may take longer.
- Premium Plan Benefits:
The premium plan includes priority customer support, access to exclusive webinars, and a dedicated success manager for enterprise users.
- Subscription Cancellation:
Customers can cancel their subscription anytime through the account settings page. No additional charges will apply after cancellation.
Answer: Shipping typically takes between 5-7 business days, depending on your location. International orders may take longer.

Sources:
- Refund Policy:
We offer a 30-day money-back guarantee on all our plans. If you're not satisfied